# Round 4 Log Comparison: final vs practice

This notebook compares the final result log (`r3_results/484904.log`) with the practice result log (`r3_test/458101.log`) by product.

Main question: why does the final result differ so much from the practice result?

Short answer from the parsed logs:

- The two runs are not the same market window: practice is day 2 from timestamp `0` to `99,900`, while final is day 3 from `0` to `999,900`.
- The strategy builds large long positions in `VEV_5100`, `VEV_5200`, and `VEV_5300`. Those products fall sharply on final day 3, creating large losses.
- `HYDROGEL_PACK` performs very well in the full final run, but it is not enough to fully offset the voucher losses.
- `VELVETFRUIT_EXTRACT` is positive in both runs and is not the main source of the gap.

In [1]:
from __future__ import annotations

import csv
import html
import io
import json
from collections import defaultdict
from pathlib import Path
from statistics import mean

try:
    from IPython.display import HTML, display
except Exception:
    HTML = None
    display = print


def find_round4_dir():
    """Support running from either the repo root or the notebook directory."""
    candidates = [
        Path.cwd(),
        Path.cwd() / "ying_research" / "Round_4",
    ]
    for candidate in candidates:
        if (candidate / "r3_results" / "484904.log").exists() and (candidate / "r3_test" / "458101.log").exists():
            return candidate
    raise FileNotFoundError("Could not find Round_4 log files from the current working directory")


ROUND4_DIR = find_round4_dir()
FINAL_LOG = ROUND4_DIR / "r3_results" / "484904.log"
PRACTICE_LOG = ROUND4_DIR / "r3_test" / "458101.log"

RUNS = {
    "final": FINAL_LOG,
    "practice": PRACTICE_LOG,
}

NUMERIC_ACTIVITY_FIELDS = {
    "day": int,
    "timestamp": int,
    "bid_price_1": float,
    "bid_volume_1": float,
    "bid_price_2": float,
    "bid_volume_2": float,
    "bid_price_3": float,
    "bid_volume_3": float,
    "ask_price_1": float,
    "ask_volume_1": float,
    "ask_price_2": float,
    "ask_volume_2": float,
    "ask_price_3": float,
    "ask_volume_3": float,
    "mid_price": float,
    "profit_and_loss": float,
}


def parse_value(value, caster):
    if value == "" or value is None:
        return None
    return caster(value)


def load_log(path: Path):
    obj = json.loads(path.read_text())
    activity_rows = []
    reader = csv.DictReader(io.StringIO(obj["activitiesLog"]), delimiter=";")
    for row in reader:
        parsed = dict(row)
        for field, caster in NUMERIC_ACTIVITY_FIELDS.items():
            parsed[field] = parse_value(parsed.get(field), caster)
        activity_rows.append(parsed)

    trades = obj.get("tradeHistory", [])
    for trade in trades:
        trade["timestamp"] = int(trade["timestamp"])
        trade["price"] = float(trade["price"])
        trade["quantity"] = float(trade["quantity"])

    return {
        "path": path,
        "submission_id": obj.get("submissionId"),
        "activity": activity_rows,
        "trades": trades,
        "logs": obj.get("logs", []),
    }


def money(x):
    if x is None:
        return ""
    return f"{x:,.2f}"


def number(x):
    if x is None:
        return ""
    if isinstance(x, float) and x.is_integer():
        return f"{int(x):,}"
    if isinstance(x, float):
        return f"{x:,.2f}"
    return f"{x:,}" if isinstance(x, int) else str(x)


def show_table(rows, columns, title=None, max_rows=None):
    rows = list(rows)
    if max_rows is not None:
        rows = rows[:max_rows]
    if title:
        display(HTML(f"<h3>{html.escape(title)}</h3>")) if HTML else print(title)
    if not rows:
        print("No rows")
        return
    if HTML:
        header = "".join(f"<th>{html.escape(str(col))}</th>" for col in columns)
        body_rows = []
        for row in rows:
            cells = []
            for col in columns:
                value = row.get(col, "")
                if isinstance(value, float):
                    value = money(value)
                cells.append(f"<td>{html.escape(str(value))}</td>")
            body_rows.append("<tr>" + "".join(cells) + "</tr>")
        display(HTML(
            "<table style='border-collapse: collapse'>"
            f"<thead><tr>{header}</tr></thead>"
            f"<tbody>{''.join(body_rows)}</tbody>"
            "</table>"
        ))
    else:
        widths = {col: max(len(str(col)), *(len(str(row.get(col, ""))) for row in rows)) for col in columns}
        print(" | ".join(str(col).ljust(widths[col]) for col in columns))
        print("-+-".join("-" * widths[col] for col in columns))
        for row in rows:
            print(" | ".join(str(row.get(col, "")).ljust(widths[col]) for col in columns))


def index_activity(rows):
    by_product = defaultdict(list)
    by_product_ts = defaultdict(dict)
    for row in rows:
        product = row["product"]
        by_product[product].append(row)
        by_product_ts[product][row["timestamp"]] = row
    for product in by_product:
        by_product[product].sort(key=lambda r: r["timestamp"])
    return by_product, by_product_ts


def final_rows_by_product(rows):
    by_product, _ = index_activity(rows)
    return {product: product_rows[-1] for product, product_rows in by_product.items()}


def first_rows_by_product(rows):
    by_product, _ = index_activity(rows)
    return {product: product_rows[0] for product, product_rows in by_product.items()}


def own_trades(trades):
    owned = []
    for trade in trades:
        is_buy = trade.get("buyer") == "SUBMISSION"
        is_sell = trade.get("seller") == "SUBMISSION"
        if not (is_buy or is_sell):
            continue
        side = 1 if is_buy else -1
        signed_qty = side * trade["quantity"]
        owned.append({
            **trade,
            "side": "buy" if is_buy else "sell",
            "signed_qty": signed_qty,
            "cash": -signed_qty * trade["price"],
            "notional": abs(signed_qty) * trade["price"],
        })
    return owned


def aggregate_own_trades(trades):
    by_symbol = defaultdict(list)
    for trade in own_trades(trades):
        by_symbol[trade["symbol"]].append(trade)
    rows = []
    for symbol, symbol_trades in sorted(by_symbol.items()):
        buy_qty = sum(t["signed_qty"] for t in symbol_trades if t["signed_qty"] > 0)
        sell_qty = -sum(t["signed_qty"] for t in symbol_trades if t["signed_qty"] < 0)
        net_qty = sum(t["signed_qty"] for t in symbol_trades)
        rows.append({
            "product": symbol,
            "trade_count": len(symbol_trades),
            "buy_qty": buy_qty,
            "sell_qty": sell_qty,
            "net_qty": net_qty,
            "cash": sum(t["cash"] for t in symbol_trades),
            "notional": sum(t["notional"] for t in symbol_trades),
            "first_ts": min(t["timestamp"] for t in symbol_trades),
            "last_ts": max(t["timestamp"] for t in symbol_trades),
        })
    return rows

runs = {name: load_log(path) for name, path in RUNS.items()}
print("Loaded:")
for name, run in runs.items():
    print(f"- {name}: {run['path']} ({len(run['activity']):,} activity rows, {len(run['trades']):,} tradeHistory rows)")

Loaded:
- final: /Users/yingzhu/Desktop/88-Pineapple/ying_research/Round_4/r3_results/484904.log (120,000 activity rows, 2,292 tradeHistory rows)
- practice: /Users/yingzhu/Desktop/88-Pineapple/ying_research/Round_4/r3_test/458101.log (12,000 activity rows, 239 tradeHistory rows)


## 1. Run-level comparison

The first important finding is that the two logs are different market samples, not a final/practice rerun over the same tape. The practice log covers only 1,000 timestamps on day 2; the final log covers 10,000 timestamps on day 3.

In [2]:
overview = []
for name, run in runs.items():
    activity = run["activity"]
    timestamps = sorted({row["timestamp"] for row in activity})
    days = sorted({row["day"] for row in activity})
    products = sorted({row["product"] for row in activity})
    final_by_product = final_rows_by_product(activity)
    overview.append({
        "run": name,
        "day": ", ".join(map(str, days)),
        "timestamp_min": min(timestamps),
        "timestamp_max": max(timestamps),
        "timestamp_count": len(timestamps),
        "activity_rows": len(activity),
        "products": len(products),
        "trade_history_rows": len(run["trades"]),
        "total_final_pnl": sum(row["profit_and_loss"] for row in final_by_product.values()),
    })

show_table(
    overview,
    ["run", "day", "timestamp_min", "timestamp_max", "timestamp_count", "activity_rows", "products", "trade_history_rows", "total_final_pnl"],
    title="Run Overview",
)

run,day,timestamp_min,timestamp_max,timestamp_count,activity_rows,products,trade_history_rows,total_final_pnl
final,3,0,999900,10000,120000,12,2292,-593.90
practice,2,0,99900,1000,12000,12,239,"5,584.29"


## 2. Product-level PnL gap

This table compares the final cumulative PnL by product at the end of each log. Positive `gap_final_minus_practice` means the final log performed better than practice for that product; negative means it performed worse.

In [3]:
final_pnl = final_rows_by_product(runs["final"]["activity"])
practice_pnl = final_rows_by_product(runs["practice"]["activity"])
products = sorted(set(final_pnl) | set(practice_pnl))

pnl_gap_rows = []
for product in products:
    final_value = final_pnl.get(product, {}).get("profit_and_loss", 0.0)
    practice_value = practice_pnl.get(product, {}).get("profit_and_loss", 0.0)
    pnl_gap_rows.append({
        "product": product,
        "practice_end_pnl": practice_value,
        "final_end_pnl": final_value,
        "gap_final_minus_practice": final_value - practice_value,
    })

pnl_gap_rows.sort(key=lambda r: r["gap_final_minus_practice"])
show_table(
    pnl_gap_rows,
    ["product", "practice_end_pnl", "final_end_pnl", "gap_final_minus_practice"],
    title="End PnL Gap by Product",
)

print("Practice total PnL:", money(sum(r["practice_end_pnl"] for r in pnl_gap_rows)))
print("Final total PnL:   ", money(sum(r["final_end_pnl"] for r in pnl_gap_rows)))
print("Total gap:        ", money(sum(r["gap_final_minus_practice"] for r in pnl_gap_rows)))

product,practice_end_pnl,final_end_pnl,gap_final_minus_practice
VEV_5200,"1,261.93","-10,823.21","-12,085.14"
VEV_5100,"1,566.22","-7,867.57","-9,433.79"
VEV_5300,669.12,"-6,091.43","-6,760.54"
VEV_4000,0.00,0.00,0.00
VEV_4500,0.00,0.00,0.00
VEV_5000,0.00,0.00,0.00
VEV_5400,0.00,0.00,0.00
VEV_5500,0.00,0.00,0.00
VEV_6000,0.00,0.00,0.00
VEV_6500,0.00,0.00,0.00


Practice total PnL: 5,584.29
Final total PnL:    -593.90
Total gap:         -6,178.18


## 3. Horizon and market-regime effect

Practice ends at timestamp `99,900`. To make the comparison fairer, the next table looks at the final day 3 PnL at the same timestamp and then at the final end. This separates two effects:

- day 2 vs day 3 behavior during the first 100k timestamps;
- the extra 900k timestamps in the final log.

In [4]:
practice_end_ts = max(row["timestamp"] for row in runs["practice"]["activity"])
final_by_product, final_by_product_ts = index_activity(runs["final"]["activity"])
practice_first = first_rows_by_product(runs["practice"]["activity"])
practice_end = final_rows_by_product(runs["practice"]["activity"])
final_first = first_rows_by_product(runs["final"]["activity"])
final_end = final_rows_by_product(runs["final"]["activity"])

horizon_rows = []
for product in products:
    final_at_practice_end = final_by_product_ts[product].get(practice_end_ts)
    if final_at_practice_end is None:
        continue
    horizon_rows.append({
        "product": product,
        "practice_pnl_99k": practice_end[product]["profit_and_loss"],
        "final_pnl_99k": final_at_practice_end["profit_and_loss"],
        "final_pnl_end": final_end[product]["profit_and_loss"],
        "final_extra_horizon_pnl": final_end[product]["profit_and_loss"] - final_at_practice_end["profit_and_loss"],
        "practice_mid_change_99k": practice_end[product]["mid_price"] - practice_first[product]["mid_price"],
        "final_mid_change_99k": final_at_practice_end["mid_price"] - final_first[product]["mid_price"],
        "final_mid_change_after_99k": final_end[product]["mid_price"] - final_at_practice_end["mid_price"],
    })

horizon_rows.sort(key=lambda r: r["final_extra_horizon_pnl"])
show_table(
    horizon_rows,
    [
        "product",
        "practice_pnl_99k",
        "final_pnl_99k",
        "final_pnl_end",
        "final_extra_horizon_pnl",
        "practice_mid_change_99k",
        "final_mid_change_99k",
        "final_mid_change_after_99k",
    ],
    title=f"Practice End vs Final Same Timestamp ({practice_end_ts:,})",
)

product,practice_pnl_99k,final_pnl_99k,final_pnl_end,final_extra_horizon_pnl,practice_mid_change_99k,final_mid_change_99k,final_mid_change_after_99k
VEV_5200,"1,261.93","-6,555.28","-10,823.21","-4,267.94",-1.50,-31.00,-18.50
VEV_5100,"1,566.22","-3,659.54","-7,867.57","-4,208.03",-3.00,-37.50,-22.50
VEV_5300,669.12,"-3,675.17","-6,091.43","-2,416.25",-3.00,-19.00,-13.00
VEV_4000,0.00,0.00,0.00,0.00,-3.50,-42.50,-21.50
VEV_4500,0.00,0.00,0.00,0.00,-3.00,-41.50,-22.00
VEV_5000,0.00,0.00,0.00,0.00,-3.00,-40.50,-22.00
VEV_5400,0.00,0.00,0.00,0.00,-1.00,-9.00,-6.00
VEV_5500,0.00,0.00,0.00,0.00,0.00,-3.50,-2.00
VEV_6000,0.00,0.00,0.00,0.00,0.00,0.00,0.00
VEV_6500,0.00,0.00,0.00,0.00,0.00,0.00,0.00


## 4. Own-trade exposure

The next cells aggregate trades where `SUBMISSION` is either buyer or seller. This exposes the position profile behind the PnL. The most important pattern is that the strategy reaches large long voucher positions very early and carries them through a falling final-day market.

In [5]:
for run_name in ["practice", "final"]:
    rows = aggregate_own_trades(runs[run_name]["trades"])
    rows.sort(key=lambda r: r["notional"], reverse=True)
    show_table(
        rows,
        ["product", "trade_count", "buy_qty", "sell_qty", "net_qty", "cash", "notional", "first_ts", "last_ts"],
        title=f"Own Trade Summary: {run_name}",
    )

product,trade_count,buy_qty,sell_qty,net_qty,cash,notional,first_ts,last_ts
HYDROGEL_PACK,43,143.00,107.00,36.00,"-356,843.00","2,495,563.00",11200,98700
VELVETFRUIT_EXTRACT,37,106.00,115.00,-9.00,"47,754.00","1,163,032.00",2700,97400
VEV_5100,29,182.00,2.00,180.00,"-30,110.00","30,832.00",1300,98600
VEV_5200,31,225.00,5.00,220.00,"-21,300.00","22,292.00",9900,98600
VEV_5300,25,183.00,4.00,179.00,"-8,305.00","8,675.00",40900,80800


product,trade_count,buy_qty,sell_qty,net_qty,cash,notional,first_ts,last_ts
HYDROGEL_PACK,582,"1,458.00","1,475.00",-17.00,"193,087.00","29,351,633.00",4400,992600
VELVETFRUIT_EXTRACT,298,800.00,782.00,18.00,"-93,054.00","8,286,868.00",10600,997500
VEV_5100,160,321.00,141.00,180.00,"-33,349.00","75,989.00",1700,993800
VEV_5200,161,359.00,139.00,220.00,"-26,163.00","47,623.00",100,993800
VEV_5300,101,259.00,79.00,180.00,"-10,804.00","16,092.00",0,942300


In [6]:
def cumulative_positions(trades):
    position = defaultdict(float)
    rows = []
    for trade in sorted(own_trades(trades), key=lambda t: (t["timestamp"], t["symbol"])):
        position[trade["symbol"]] += trade["signed_qty"]
        rows.append({
            "timestamp": trade["timestamp"],
            "product": trade["symbol"],
            "side": trade["side"],
            "price": trade["price"],
            "qty": abs(trade["signed_qty"]),
            "position_after": position[trade["symbol"]],
        })
    return rows

final_positions = cumulative_positions(runs["final"]["trades"])
practice_positions = cumulative_positions(runs["practice"]["trades"])

limit_like_targets = {"VEV_5100": 180, "VEV_5200": 220, "VEV_5300": 180}
reach_rows = []
for run_name, position_rows in [("practice", practice_positions), ("final", final_positions)]:
    for product, target in limit_like_targets.items():
        reached = next((row for row in position_rows if row["product"] == product and abs(row["position_after"]) >= target), None)
        reach_rows.append({
            "run": run_name,
            "product": product,
            "target_abs_position": target,
            "first_reach_ts": reached["timestamp"] if reached else None,
            "position_after": reached["position_after"] if reached else None,
        })

show_table(reach_rows, ["run", "product", "target_abs_position", "first_reach_ts", "position_after"], title="When Large Voucher Positions Were Reached")

run,product,target_abs_position,first_reach_ts,position_after
practice,VEV_5100,180,76200,180.00
practice,VEV_5200,220,46500,220.00
practice,VEV_5300,180,57300,180.00
final,VEV_5100,180,22800,180.00
final,VEV_5200,220,12800,220.00
final,VEV_5300,180,900,180.00


## 5. Visual diagnostics

The charts below use lightweight HTML/SVG, so they do not require `pandas`, `matplotlib`, or any other plotting package. They cover the most useful views for explaining the result gap:

- end PnL gap by product;
- final extra-horizon contribution after the practice window ends;
- PnL curves for the products that actually matter;
- mid-price regime shift;
- cumulative own-position paths;
- trade notional and final net exposure.

In [7]:
def sample_points(points, max_points=700):
    if len(points) <= max_points:
        return points
    step = max(1, len(points) // max_points)
    sampled = points[::step]
    if sampled[-1] != points[-1]:
        sampled.append(points[-1])
    return sampled


def _fmt_axis(value):
    if abs(value) >= 1000:
        return f"{value / 1000:.0f}k"
    return f"{value:.0f}"


def _scale(value, lo, hi, out_lo, out_hi):
    if hi == lo:
        return (out_lo + out_hi) / 2
    return out_lo + (value - lo) * (out_hi - out_lo) / (hi - lo)


def show_svg(svg):
    if HTML:
        display(HTML(svg))
    else:
        print("SVG chart requires an IPython/Jupyter display.")


def bar_chart(rows, label_key, value_key, title, width=980, row_h=28, left=190, right=120):
    rows = list(rows)
    values = [float(row[value_key]) for row in rows]
    vmin = min(0.0, min(values))
    vmax = max(0.0, max(values))
    if vmin == vmax:
        vmin -= 1
        vmax += 1
    top = 44
    bottom = 28
    height = top + bottom + row_h * len(rows)
    plot_w = width - left - right
    zero_x = _scale(0, vmin, vmax, left, left + plot_w)
    parts = [
        f"<div style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; margin: 18px 0'>",
        f"<h3 style='margin: 0 0 8px 0'>{html.escape(title)}</h3>",
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        f"<line x1='{zero_x:.1f}' y1='{top - 12}' x2='{zero_x:.1f}' y2='{height - bottom + 4}' stroke='#888' stroke-width='1'/>",
    ]
    for i, row in enumerate(rows):
        y = top + i * row_h
        value = float(row[value_key])
        x = _scale(value, vmin, vmax, left, left + plot_w)
        bar_x = min(x, zero_x)
        bar_w = max(1, abs(x - zero_x))
        color = "#2e7d32" if value >= 0 else "#c62828"
        parts.extend([
            f"<text x='8' y='{y + 17}' font-size='12' fill='#222'>{html.escape(str(row[label_key]))}</text>",
            f"<rect x='{bar_x:.1f}' y='{y + 5}' width='{bar_w:.1f}' height='16' fill='{color}' opacity='0.82'/>",
            f"<text x='{left + plot_w + 8}' y='{y + 17}' font-size='12' fill='#222'>{html.escape(money(value))}</text>",
        ])
    parts.extend([
        f"<text x='{left}' y='{height - 8}' font-size='11' fill='#666'>{html.escape(_fmt_axis(vmin))}</text>",
        f"<text x='{zero_x + 4:.1f}' y='{height - 8}' font-size='11' fill='#666'>0</text>",
        f"<text x='{left + plot_w - 32}' y='{height - 8}' font-size='11' fill='#666'>{html.escape(_fmt_axis(vmax))}</text>",
        "</svg></div>",
    ])
    show_svg("".join(parts))


def grouped_bar_chart(rows, label_key, series, title, width=980, row_h=34, left=190, right=170):
    rows = list(rows)
    values = [float(row[key]) for row in rows for key, _label, _color in series]
    vmin = min(0.0, min(values))
    vmax = max(0.0, max(values))
    if vmin == vmax:
        vmin -= 1
        vmax += 1
    top = 58
    bottom = 30
    height = top + bottom + row_h * len(rows)
    plot_w = width - left - right
    zero_x = _scale(0, vmin, vmax, left, left + plot_w)
    bar_h = 10
    parts = [
        f"<div style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; margin: 18px 0'>",
        f"<h3 style='margin: 0 0 8px 0'>{html.escape(title)}</h3>",
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        f"<line x1='{zero_x:.1f}' y1='{top - 14}' x2='{zero_x:.1f}' y2='{height - bottom + 4}' stroke='#888' stroke-width='1'/>",
    ]
    legend_x = left
    for _, label, color in series:
        parts.append(f"<rect x='{legend_x}' y='22' width='10' height='10' fill='{color}' opacity='0.82'/>")
        parts.append(f"<text x='{legend_x + 16}' y='31' font-size='12' fill='#333'>{html.escape(label)}</text>")
        legend_x += 150
    for i, row in enumerate(rows):
        y = top + i * row_h
        parts.append(f"<text x='8' y='{y + 20}' font-size='12' fill='#222'>{html.escape(str(row[label_key]))}</text>")
        for j, (key, _label, color) in enumerate(series):
            value = float(row[key])
            x = _scale(value, vmin, vmax, left, left + plot_w)
            bar_x = min(x, zero_x)
            bar_w = max(1, abs(x - zero_x))
            yy = y + 4 + j * (bar_h + 3)
            parts.append(f"<rect x='{bar_x:.1f}' y='{yy}' width='{bar_w:.1f}' height='{bar_h}' fill='{color}' opacity='0.82'/>")
        label_text = " / ".join(money(float(row[key])) for key, _label, _color in series)
        parts.append(f"<text x='{left + plot_w + 8}' y='{y + 20}' font-size='12' fill='#222'>{html.escape(label_text)}</text>")
    parts.extend([
        f"<text x='{left}' y='{height - 8}' font-size='11' fill='#666'>{html.escape(_fmt_axis(vmin))}</text>",
        f"<text x='{zero_x + 4:.1f}' y='{height - 8}' font-size='11' fill='#666'>0</text>",
        f"<text x='{left + plot_w - 32}' y='{height - 8}' font-size='11' fill='#666'>{html.escape(_fmt_axis(vmax))}</text>",
        "</svg></div>",
    ])
    show_svg("".join(parts))


def line_chart(series, title, y_label, width=980, height=260, left=72, right=24, top=42, bottom=38, include_zero=True):
    prepared = []
    for item in series:
        points = [(float(x), float(y)) for x, y in item["points"] if x is not None and y is not None]
        if points:
            prepared.append({**item, "points": sample_points(points)})
    if not prepared:
        return
    all_x = [x for item in prepared for x, _ in item["points"]]
    all_y = [y for item in prepared for _, y in item["points"]]
    xmin, xmax = min(all_x), max(all_x)
    ymin, ymax = min(all_y), max(all_y)
    if include_zero:
        ymin = min(ymin, 0.0)
        ymax = max(ymax, 0.0)
    pad = (ymax - ymin) * 0.08 or 1
    ymin -= pad
    ymax += pad
    plot_w = width - left - right
    plot_h = height - top - bottom
    zero_y = _scale(0, ymin, ymax, top + plot_h, top)
    parts = [
        f"<div style='font-family: -apple-system, BlinkMacSystemFont, Segoe UI, sans-serif; margin: 18px 0'>",
        f"<h3 style='margin: 0 0 8px 0'>{html.escape(title)}</h3>",
        f"<svg width='{width}' height='{height}' viewBox='0 0 {width} {height}'>",
        f"<rect x='{left}' y='{top}' width='{plot_w}' height='{plot_h}' fill='white' stroke='#ddd'/>",
        f"<line x1='{left}' y1='{zero_y:.1f}' x2='{left + plot_w}' y2='{zero_y:.1f}' stroke='#999' stroke-width='1'/>",
        f"<text x='8' y='{top + 12}' font-size='11' fill='#666'>{html.escape(y_label)}</text>",
    ]
    legend_x = left + 8
    for item in prepared:
        color = item.get("color", "#1565c0")
        pts = []
        for x, y in item["points"]:
            sx = _scale(x, xmin, xmax, left, left + plot_w)
            sy = _scale(y, ymin, ymax, top + plot_h, top)
            pts.append(f"{sx:.1f},{sy:.1f}")
        parts.append(f"<polyline points='{' '.join(pts)}' fill='none' stroke='{color}' stroke-width='2'/>")
        parts.append(f"<rect x='{legend_x}' y='20' width='10' height='10' fill='{color}'/>")
        parts.append(f"<text x='{legend_x + 15}' y='29' font-size='12' fill='#333'>{html.escape(item['label'])}</text>")
        legend_x += 150
    for x_value, label in [(xmin, _fmt_axis(xmin)), (xmax, _fmt_axis(xmax))]:
        sx = _scale(x_value, xmin, xmax, left, left + plot_w)
        parts.append(f"<text x='{sx - 16:.1f}' y='{height - 10}' font-size='11' fill='#666'>{html.escape(label)}</text>")
    parts.extend([
        f"<text x='{left + 4}' y='{top + 12}' font-size='11' fill='#666'>{html.escape(_fmt_axis(ymax))}</text>",
        f"<text x='{left + 4}' y='{top + plot_h - 4}' font-size='11' fill='#666'>{html.escape(_fmt_axis(ymin))}</text>",
        "</svg></div>",
    ])
    show_svg("".join(parts))


def product_series(run_name, product, field):
    by_product, _ = index_activity(runs[run_name]["activity"])
    return [(row["timestamp"], row[field]) for row in by_product.get(product, [])]


def position_series(position_rows, product):
    points = [(0, 0.0)]
    points.extend((row["timestamp"], row["position_after"]) for row in position_rows if row["product"] == product)
    return points


key_products = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT", "VEV_5100", "VEV_5200", "VEV_5300"]
traded_products = ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT", "VEV_5100", "VEV_5200", "VEV_5300"]

bar_chart(pnl_gap_rows, "product", "gap_final_minus_practice", "End PnL Gap: Final minus Practice")
bar_chart(horizon_rows, "product", "final_extra_horizon_pnl", "Final Extra-Horizon PnL After Practice Window")
grouped_bar_chart(
    horizon_rows,
    "product",
    [
        ("practice_mid_change_99k", "practice first 99k", "#6a1b9a"),
        ("final_mid_change_99k", "final first 99k", "#1565c0"),
        ("final_mid_change_after_99k", "final after 99k", "#ef6c00"),
    ],
    "Mid-Price Change by Product and Window",
)

for product in key_products:
    line_chart(
        [
            {"label": "practice day 2", "points": product_series("practice", product, "profit_and_loss"), "color": "#6a1b9a"},
            {"label": "final day 3", "points": product_series("final", product, "profit_and_loss"), "color": "#1565c0"},
        ],
        f"PnL Path: {product}",
        "PnL",
    )

for product in ["VEV_5100", "VEV_5200", "VEV_5300"]:
    line_chart(
        [
            {"label": "practice position", "points": position_series(practice_positions, product), "color": "#6a1b9a"},
            {"label": "final position", "points": position_series(final_positions, product), "color": "#1565c0"},
        ],
        f"Cumulative Own Position: {product}",
        "position",
    )

final_trade_rows = aggregate_own_trades(runs["final"]["trades"])
practice_trade_rows = aggregate_own_trades(runs["practice"]["trades"])
trade_compare = []
for product in traded_products:
    final_row = next(row for row in final_trade_rows if row["product"] == product)
    practice_row = next(row for row in practice_trade_rows if row["product"] == product)
    trade_compare.append({
        "product": product,
        "practice_notional": practice_row["notional"],
        "final_notional": final_row["notional"],
        "practice_net_qty": practice_row["net_qty"],
        "final_net_qty": final_row["net_qty"],
    })

grouped_bar_chart(
    trade_compare,
    "product",
    [
        ("practice_notional", "practice notional", "#6a1b9a"),
        ("final_notional", "final notional", "#1565c0"),
    ],
    "Own-Trade Notional by Product",
)

grouped_bar_chart(
    trade_compare,
    "product",
    [
        ("practice_net_qty", "practice net qty", "#6a1b9a"),
        ("final_net_qty", "final net qty", "#1565c0"),
    ],
    "Ending Own Net Quantity by Product",
)

## 6. Price vs timestamp by product type

These charts compare `mid_price` against timestamp for practice and final, separated by product type. The x-axis uses each log's actual timestamp, so practice stops at `99,900` while final continues to `999,900`.

The most useful comparison is the early-window shape: practice day 2 is mostly flat for the traded vouchers, while final day 3 reprices downward much more aggressively and then keeps drifting lower.

In [8]:
PRICE_PRODUCT_GROUPS = {
    "Core products": ["HYDROGEL_PACK", "VELVETFRUIT_EXTRACT"],
    "VEV traded strikes": ["VEV_5100", "VEV_5200", "VEV_5300"],
    "VEV other visible strikes": ["VEV_4000", "VEV_4500", "VEV_5000", "VEV_5400", "VEV_5500", "VEV_6000", "VEV_6500"],
}

for group_name, group_products in PRICE_PRODUCT_GROUPS.items():
    if HTML:
        display(HTML(f"<h3>{html.escape(group_name)}</h3>"))
    else:
        print(group_name)
    for product in group_products:
        line_chart(
            [
                {"label": "practice day 2 mid", "points": product_series("practice", product, "mid_price"), "color": "#6a1b9a"},
                {"label": "final day 3 mid", "points": product_series("final", product, "mid_price"), "color": "#1565c0"},
            ],
            f"Mid Price vs Timestamp: {product}",
            "mid price",
            include_zero=False,
        )

## 6. Product-by-product interpretation

### `HYDROGEL_PACK`

`HYDROGEL_PACK` is the strongest final product. Practice ends at only about 100k timestamps and shows a modest gain, while final has the full day 3 horizon and ends with a very large positive PnL. This product explains why the final run is not deeply negative despite the voucher losses.

### `VEV_5100`, `VEV_5200`, `VEV_5300`

These are the main reason the final and practice results differ. The strategy builds large long positions in all three products. In the practice window, day 2 voucher mids barely move over the first 100k timestamps, so the strategy can still look profitable. On final day 3, the same strikes fall much more, both inside the first 100k timestamps and again over the rest of the full-day horizon. The carried long exposure turns that market move into large losses.

### `VELVETFRUIT_EXTRACT`

`VELVETFRUIT_EXTRACT` is positive in both runs and has much smaller contribution than `HYDROGEL_PACK` or the voucher basket. It helps the final result but does not explain the large gap.

### Inactive voucher strikes

`VEV_4000`, `VEV_4500`, `VEV_5000`, `VEV_5400`, `VEV_5500`, `VEV_6000`, and `VEV_6500` have zero PnL in both logs. They are visible in market data but are not materially traded by the submitted strategy.

## Bottom line

The final result is not simply a worse version of the practice result. It is a different day and a 10x longer horizon. The strategy's hydrogel market making scales well over the longer final run, but the voucher component is regime-sensitive: it accumulates large long exposure in `VEV_5100/5200/5300`, and final day 3 is a falling voucher market. That product-level mismatch is the main cause of the large difference.

## 7. 中文结论：PnL 差异由什么驱动，以及 Round 4 怎么改

### 1. 核心驱动：voucher 仓位不对称 + 标的趋势性下跌

两份日志的产品级 PnL 差异很集中：

- `HYDROGEL_PACK`: practice 约 `+1,710`，final 约 `+23,066`，差额约 `+21,356`。
- `VELVETFRUIT_EXTRACT`: practice 约 `+377`，final 约 `+1,122`，差额约 `+745`。
- `VEV_5100`: practice 约 `+1,566`，final 约 `-7,868`，差额约 `-9,434`。
- `VEV_5200`: practice 约 `+1,262`，final 约 `-10,823`，差额约 `-12,085`。
- `VEV_5300`: practice 约 `+669`，final 约 `-6,091`，差额约 `-6,761`。
- 其余 VEV strikes 基本没有 PnL。

合计上，practice 约 `+5,584`，final 约 `-594`，总差额约 `-6,178`。表面上 final 只是比 practice 少了约 6k，但内部结构很不一样：`HYDROGEL_PACK` 在 final 多赚了约 21k，而三只 traded vouchers 合计多亏了约 28k，最终互相抵消后才得到这个总差额。

时间窗口也不同：practice 是 day 2 的 `0` 到 `99,900`，final 是 day 3 的 `0` 到 `999,900`。但关键不是只有窗口长度。即使对齐到 timestamp `99,900`，final 的 `VEV_5100/5200/5300` 已经大幅亏损，而 practice 同一窗口是盈利的。原因是 day 3 的 `VELVETFRUIT_EXTRACT` 和相关 voucher mids 下跌更剧烈，策略又持有大量 voucher 多头。

### 2. 直接亏损机制：voucher 很早拉满多头，之后没有有效降仓

own-trade 聚合显示，final 中三只 traded voucher 都是明显净多：

- `VEV_5100`: buy `321`，sell `141`，net `+180`，在 timestamp `22,800` 已达到大仓位。
- `VEV_5200`: buy `359`，sell `139`，net `+220`，在 timestamp `12,800` 已达到大仓位。
- `VEV_5300`: buy `259`，sell `79`，net `+180`，在 timestamp `900` 就达到大仓位。

这说明 final day 3 里，策略非常早就把 voucher 多头仓位打满，然后在持续下跌的市场里持有到后面。亏损主要不是最后一小段突然发生，而是「早期建多 + 全天下跌 + 出仓不足」叠加出来的。

### 3. 策略层原因：voucher fair value 太静态，容易系统性高估

提交代码里 voucher 的公允价主要来自固定回归系数：

```python
VEV_REG_COEFS = {
    5100: (-3950.957951, 0.784321),
    5200: (-2871.358698, 0.565115),
    5300: (-1704.686347, 0.333603),
}
VEV_REG_OPEN_EDGE = {
    5100: 0.80,
    5200: 0.70,
    5300: 0.75,
}
VEV_REG_CLOSE_EDGE = {
    5100: 0.20,
    5200: 0.65,
    5300: 0.20,
}
```

这种模型的问题是：

- 它主要用静态线性关系估 voucher value，没有充分跟随 day、TTE、IV surface、underlying regime。
- 当 final day 3 的 underlying 和 vouchers 持续下跌时，模型容易认为 market price 偏低，于是继续买。
- close edge / exit 机制偏弱，尤其 `VEV_5200` 的 close edge 接近 open edge，仓位不容易被及时减掉。
- 三只 traded voucher 的 buy/sell 比例明显偏买，说明信号存在系统性 long bias。

换句话说，voucher 子策略在 practice 的短窗口里看起来有效，但在 final day 3 的下跌 regime 里暴露出「静态估值 + 单边加仓 + 缺少 hedge」的问题。

### 4. 为什么 `HYDROGEL_PACK` 表现很好

`HYDROGEL_PACK` 是更接近 market-making 的逻辑。final 中它成交很多，但 buy/sell 很对称：buy `1458`，sell `1475`，期末净仓只有 `-17`。这说明它不是靠赌方向赚钱，而是靠双边成交、库存回正、长窗口流量累积赚钱。

这和 voucher 形成鲜明对比：

- `HYDROGEL_PACK`：高成交、低净仓、强 inventory control，final 长窗口放大盈利。
- `VEV_5100/5200/5300`：低成交但高净多，方向错时亏损被放大。

所以 Round 4 不应该优先重写 `HYDROGEL_PACK`，它反而是组合里最稳的部分。主要需要修 voucher 部分。

### 5. Round 4 改进建议

#### A. 给 voucher 加 delta hedge，优先级最高

当前最大问题是 voucher 多头直接暴露在 `VELVETFRUIT_EXTRACT` 下跌里。可以用回归斜率作为近似 delta：

```python
hedge_qty = -sum(voucher_position[strike] * VEV_REG_COEFS[strike][1] for strike in active_strikes)
```

然后用 `VELVETFRUIT_EXTRACT` 做反向 hedge。即使 hedge 不完美，也能显著降低 day 3 这种趋势下跌对 voucher book 的冲击。

#### B. 让 voucher fair value 动态化

静态回归要改成随市场状态更新。可以考虑：

- 加入当前 `VELVETFRUIT_EXTRACT` mid price。
- 加入 day / time-to-expiry。
- 用 implied volatility 或 moneyness 做特征。
- 至少按 day 重新校准不同的 intercept/slope，不要 day 2 参数直接套 day 3。

目标是避免在下跌 regime 中反复把真实 repricing 误判成 mispricing。

#### C. 限制 voucher 加仓速度和最大净 delta

final 中 `VEV_5300` 在 timestamp `900` 就达到 `+180`，太早也太满。建议增加：

- 每个 strike 的单窗口最大净买入量。
- 组合层面的最大 net voucher delta。
- 越接近 max position，open threshold 越高。
- 当 underlying 短窗口趋势向下时，暂停新增 long voucher。

这比单纯降低 position limit 更好，因为它控制的是「建仓速度」和「组合方向暴露」。

#### D. 加 drawdown / regime circuit breaker

当单个 voucher 或 voucher basket 出现明显亏损时，应该停止加仓甚至减仓。例如：

- 单 strike PnL < `-2,000` 后停止新增 long。
- voucher basket PnL < `-5,000` 后只允许减仓，不允许加仓。
- `VELVETFRUIT_EXTRACT` 在最近 N 个 timestamp 下跌超过阈值时，暂停 long voucher signal。

final 中三只 traded voucher 早就进入亏损，但策略仍保持大多头，这是需要避免的。

#### E. 买卖信号要更对称

现在结果显示策略明显更容易买 voucher，而不是卖或平。可以把信号改成围绕 fair value 的对称 z-score：

- market ask < fair - threshold 才买；
- market bid > fair + threshold 才卖；
- threshold 用 rolling residual volatility 调整，而不是固定 `0.70/0.80`。

这样可以减少模型系统性高估时一路接刀子的风险。

#### F. 保留 `HYDROGEL_PACK` 主逻辑，谨慎调参

`HYDROGEL_PACK` final 贡献很强，说明它的 inventory penalty 和 market-making 参数大体有效。可以小调 spread / take edge，但不要为了修 voucher 问题重构 hydrogel。真正需要风险控制的是 voucher book。

### 6. 最终判断

final 和 practice 差异的本质是：

> practice 是较短、较温和的 day 2 window；final 是更长、且 voucher/underlying 明显下跌的 day 3 window。策略的 `HYDROGEL_PACK` 做市逻辑适应长窗口并赚到很多，但 voucher 子策略在静态 fair value、单边多头、缺少 hedge 和缺少止损的组合下，把 day 3 的下跌转化成了大亏。

Round 4 最应该改的是 voucher risk model：动态 fair value、delta hedge、加仓速度限制、drawdown circuit breaker、对称买卖信号。`HYDROGEL_PACK` 可以保留为主要正贡献模块。